In [ ]:
def read_band(city_dir, band):
    path = list(city_dir.glob(f"*.{band}.tif"))[0]
    with rasterio.open(path) as src:
        return src.read(1).astype(np.float32)

b2 = read_band(city_dir, "SR_B2")
b3 = read_band(city_dir, "SR_B3")
b4 = read_band(city_dir, "SR_B4")
b10 = read_band(city_dir, "ST_B10")

def normalize(img):
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip(img, p2, p98)
    return (img - p2) / (p98 - p2 + 1e-8)

rgb = np.dstack([
    normalize(b4),
    normalize(b3),
    normalize(b2)
])

thermal = normalize(b10)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(rgb)
plt.title("True Color RGB: B4-B3-B2")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(thermal, cmap="gray")
plt.title("Thermal IR: ST_B10")
plt.axis("off")

plt.show()

In [ ]:
b5 = read_band(city_dir, "SR_B5")
b6 = read_band(city_dir, "SR_B6")

# NDVI
ndvi = (b5 - b4) / (b5 + b4 + 1e-8)

# NDWI
ndwi = (b3 - b5) / (b3 + b5 + 1e-8)

# NDBI
ndbi = (b6 - b5) / (b6 + b5 + 1e-8)

print("NDVI:", ndvi.min(), ndvi.max())
print("NDWI:", ndwi.min(), ndwi.max())
print("NDBI:", ndbi.min(), ndbi.max())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im1 = axes[0].imshow(ndvi, cmap="RdYlGn")
axes[0].set_title("NDVI (Vegetation)")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(ndwi, cmap="Blues")
axes[1].set_title("NDWI (Water)")
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(ndbi, cmap="inferno")
axes[2].set_title("NDBI (Built-up)")
axes[2].axis("off")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import rasterio

RAW_DIR = Path("../data/raw")

for city_dir in sorted(RAW_DIR.iterdir()):
    if city_dir.is_dir():

        thermal_file = list(city_dir.glob("*.ST_B10.tif"))[0]

        with rasterio.open(thermal_file) as src:
            h, w = src.shape

        print(city_dir.name, "->", h, "x", w)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(ndvi.flatten(), bins=100)
plt.title("NDVI Distribution")
plt.xlabel("NDVI")
plt.ylabel("Pixels")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(b10.flatten(), bins=100)
plt.title("Thermal Distribution")
plt.xlabel("Temperature")
plt.ylabel("Pixels")
plt.show()